[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_06_Gradient_Descent_SGD_Momentum_and_Adam.ipynb)


# Experiment 6: Gradient Descent, SGD, Momentum and Adam

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 6 – Gradient Descent, SGD, Momentum and Adam

In [ ]:
# Optimiser paths on f(x,y)=x^2+5y^2
def grad(z): return np.array([2*z[0],10*z[1]])
start=np.array([4.,3.]); steps=35; lr=.12
paths={}
for method in ["GD","SGD","Momentum","Adam"]:
    z=start.copy(); v=np.zeros(2); m=np.zeros(2); vv=np.zeros(2); path=[z.copy()]
    for t in range(1,steps+1):
        g=grad(z)
        if method=="SGD": g=g+np.random.normal(0,.3,2)
        if method=="Momentum": v=.85*v+g; z-=lr*v
        elif method=="Adam":
            m=.9*m+.1*g; vv=.999*vv+.001*g*g
            z-=.25*(m/(1-.9**t))/(np.sqrt(vv/(1-.999**t))+1e-8)
        else: z-=lr*g
        path.append(z.copy())
    paths[method]=np.array(path)
xx,yy=np.meshgrid(np.linspace(-4,4,100),np.linspace(-3,3,100)); zz=xx**2+5*yy**2
plt.contour(xx,yy,zz,levels=25)
for name,path in paths.items(): plt.plot(path[:,0],path[:,1],"-o",ms=2,label=name)
plt.legend(); plt.title("Optimiser paths"); plt.show()
# Keras comparison (small synthetic regression)
import tensorflow as tf
tf.random.set_seed(42)
X=np.random.randn(1000,4).astype("float32"); y=(X@np.array([2.,-1.,.5,3.])+np.random.randn(1000)*.2).astype("float32")
optimizers={"SGD":tf.keras.optimizers.SGD(.01),"Momentum":tf.keras.optimizers.SGD(.01,momentum=.9),"Adam":tf.keras.optimizers.Adam(.01)}
histories={}
for name,opt in optimizers.items():
    model=tf.keras.Sequential([tf.keras.layers.Input((4,)),tf.keras.layers.Dense(16,activation="relu"),tf.keras.layers.Dense(1)])
    model.compile(opt,"mse"); h=model.fit(X,y,epochs=25,batch_size=32,verbose=0,validation_split=.2)
    histories[name]=h.history["loss"]
for n,v in histories.items(): plt.plot(v,label=n)
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("Keras optimiser comparison"); plt.show()